In [2]:
import numpy as np

# 2.1 Gradient tekshiruvi (gradient check)
x  = np.array([2., 1.])
W1 = np.array([[1., -1.], [1., 2.]])
b1 = np.array([0., -1.])
w2 = np.array([1., 3.])
b2 = 0.0
y  = 4.0

def forward(x, W1):
    z1 = x @ W1 + b1
    h = np.maximum(0, z1)
    return z1, h, h @ w2 + b2

def backward(x, y):
    z1, h, y_hat = forward(x, W1)
    d = 2 * (y_hat - y)
    dz1 = (w2 * d) * (z1 > 0)
    dW1 = np.outer(x, dz1)
    return dW1

def loss_fn(W):
    y_hat = forward(x, W)[2]
    return (y_hat - y) ** 2

def numerical_grad(W1, eps=1e-5):
    g = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            W_plus = W1.copy()
            W_plus[i, j] += eps
            W_minus = W1.copy()
            W_minus[i, j] -= eps
            g[i, j] = (loss_fn(W_plus) - loss_fn(W_minus)) / (2 * eps)
    return g

dW1 = backward(x, y)
num = numerical_grad(W1)
farq = np.max(np.abs(dW1 - num))

print(" -- 2.1 --")
print("backward() dW1:\n", dW1)
print("numerical_grad:\n", num)
print("Eng katta farq:", farq)
print("Test o'tdi!" if farq < 1e-6 else "Xato bor!")

# 2.1 XULOSA
# backward() va numerical_grad() bir xil gradient berdi.
# Eng katta farq taxminan 2.3e-10, bu 1e-6 dan ancha kichik.


# MLP klassi (05_mlp.py) + ReLU varianti (2.4 uchun)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

class MLP:
    def __init__(self, n_in, n_h, seed=0, relu=False):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 1, (n_in, n_h))
        self.W2 = r.normal(0, 1, (n_h, 1))
        self.b1, self.b2 = np.zeros(n_h), np.zeros(1)
        self.relu = relu
    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        if self.relu:
            self.h = np.maximum(0, self.z1)
        else:
            self.h = sigmoid(self.z1)
        self.p = sigmoid(self.h @ self.W2 + self.b2)
        return self.p

    def step(self, X, y, lr):
        dz2 = (self.p - y) / len(X)
        if self.relu:
            dz1 = dz2 @ self.W2.T * (self.z1 > 0)
        else:
            dz1 = dz2 @ self.W2.T * self.h * (1 - self.h)
        self.W2 -= lr * self.h.T @ dz2
        self.b2 -= lr * dz2.sum(0)
        self.W1 -= lr * X.T @ dz1
        self.b1 -= lr * dz1.sum(0)


X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([[0], [1], [1], [0]], dtype=float)

def train(lr, epochs, n_h=4, relu=False):
    net = MLP(2, n_h, seed=1, relu=relu)
    for ep in range(epochs):
        net.forward(X)
        net.step(X, Y, lr)
    p = net.forward(X)
    loss = -np.mean(Y * np.log(p) + (1 - Y) * np.log(1 - p))
    bashorat = (p > 0.5).astype(int).ravel()
    togri = (bashorat == Y.ravel()).all()
    return round(loss, 4), bashorat, togri


# 2.2 Learning rate tanlash (2000 epoch)

print("\n -- 2.2 --")
for lr in [0.1, 0.5, 1.0, 2.0, 5.0]:
    loss, bashorat, togri = train(lr, 2000)
    print(f"lr={lr}: loss={loss}, bashorat={bashorat}, to'g'ri={togri}")

# 2.2 XULOSA
# | lr  | Loss (epoch 2000) | Bashorat  | To'g'ri ishlaydimi? ||
# | 0.1 | 0.6169            | [0 1 0 0] | Yo'q                |
# | 0.5 | 0.0140            | [0 1 1 0] | Ha                  |
# | 1.0 | 0.0049            | [0 1 1 0] | Ha                  |
# | 2.0 | 0.0020            | [0 1 1 0] | Ha                  |
# | 5.0 | 0.0007            | [0 1 1 0] | Ha                  |
#
# 2000 epochda qoidani o'rganadigan lr lar: 0.5, 1.0, 2.0, 5.0.
#
# Savol: nega bu yerda katta lr yaxshiroq?
# 1-qismda gradientlar katta edi (masalan 24), shuning uchun katta lr
# bilan qadam minimumdan oshib ketardi. Bu yerda esa sigmoid + BCE
# gradientlari juda kichik: sigmoid hosilasi eng ko'pi bilan 0.25 va
# gradient yana 4 ga bo'linadi. Shuning uchun kichik lr bilan loss juda
# sekin tushadi va 2000 epoch yetmaydi. "Katta" yoki "kichik" lr doim
# gradient kattaligiga nisbatan aytiladi.


# 2.3 Yashirin neyronlar soni (lr=1.0, 5000 epoch)

print("\n -- 2.3 --")
for n_h in [1, 2, 4, 8]:
    loss, bashorat, togri = train(1.0, 5000, n_h=n_h)
    print(f"n_h={n_h}: loss={loss}, bashorat={bashorat}, to'g'ri={togri}")

# ----------------- 2.3 XULOSA -----------------
# | n_h | Loss   | Bashorat  | To'g'ri? |
# | 1   | 0.4786 | [0 1 1 1] | Yo'q     |
# | 2   | 0.0020 | [0 1 1 0] | Ha       |
# | 4   | 0.0016 | [0 1 1 0] | Ha       |
# | 8   | 0.0010 | [0 1 1 0] | Ha       |
#
# Chipga kamida 2 ta yashirin neyron kerak.
#
# n_h = 1 bo'lsa, tarmoq faqat bitta to'g'ri chiziq o'tkaza oladi,
# XOR esa chiziqli ajralmaydi. Natijada u OR ga o'xshab qoladi
# ([0 1 1 1]). Ikkita neyron ikkita chiziq o'tkazadi.


# 2.4 ReLU va sigmoid (5000 epoch)
print("\n --  2.4 --")
for lr in [0.1, 0.5, 1.0]:
    loss_r, b_r, t_r = train(lr, 5000, relu=True)
    loss_s, b_s, t_s = train(lr, 5000, relu=False)
    print(f"lr={lr}: ReLU loss={loss_r} {b_r} | Sigmoid loss={loss_s} {b_s}")

# ----------------- 2.4 XULOSA -----------------
# | lr  | ReLU loss | Sigmoid loss | Bashorat (ikkalasi) |
# | 0.1 | 0.0015    | 0.0711       | [0 1 1 0]           |
# | 0.5 | 0.0002    | 0.0037       | [0 1 1 0]           |
# | 1.0 | 0.0001    | 0.0016       | [0 1 1 0]           |
#
# seed=1 da ReLU versiyasi har bir lr da XOR ni to'g'ri o'rgandi va
# sigmoid dan ancha past loss berdi. lr=0.1 da ham o'rgandi (sigmoid
# 2.2 da shu lr da o'rgana olmagan edi).  Chunki ReLU hosilasi 1 ga
# teng, sigmoid niki esa eng ko'pi bilan 0.25, shuning uchun gradient
# kuchliroq o'tadi.
#
# Chipga qo'ysa bo'ladimi? Ha, mumkin: exp() kerak emas, tezroq ishlaydi
# va natija yaxshi. Lekin ehtiyot bo'lish kerak: ReLU da z < 0 bo'lsa
# gradient 0 bo'ladi ("o'lik neyron"), shuning uchun boshqa seed larda
# ba'zan o'rganmay qolishi mumkin. Chipga faqat ishlashi tekshirilgan
# vaznlarni yuklash kerak.


 -- 2.1 --
backward() dW1:
 [[-4. -0.]
 [-2. -0.]]
numerical_grad:
 [[-4.  0.]
 [-2.  0.]]
Eng katta farq: 2.6204816094832495e-11
Test o'tdi!

 -- 2.2 --
lr=0.1: loss=0.6169, bashorat=[0 1 0 0], to'g'ri=False
lr=0.5: loss=0.014, bashorat=[0 1 1 0], to'g'ri=True
lr=1.0: loss=0.0049, bashorat=[0 1 1 0], to'g'ri=True
lr=2.0: loss=0.002, bashorat=[0 1 1 0], to'g'ri=True
lr=5.0: loss=0.0007, bashorat=[0 1 1 0], to'g'ri=True

 -- 2.3 --
n_h=1: loss=0.4786, bashorat=[0 1 1 1], to'g'ri=False
n_h=2: loss=0.002, bashorat=[0 1 1 0], to'g'ri=True
n_h=4: loss=0.0016, bashorat=[0 1 1 0], to'g'ri=True
n_h=8: loss=0.001, bashorat=[0 1 1 0], to'g'ri=True

 --  2.4 --
lr=0.1: ReLU loss=0.0015 [0 1 1 0] | Sigmoid loss=0.0711 [0 1 1 0]
lr=0.5: ReLU loss=0.0002 [0 1 1 0] | Sigmoid loss=0.0037 [0 1 1 0]
lr=1.0: ReLU loss=0.0001 [0 1 1 0] | Sigmoid loss=0.0016 [0 1 1 0]


In [3]:
import torch
import torch.nn as nn
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


#  3.1
x  = torch.tensor([2., 1.])
W1 = torch.tensor([[1., -1.], [1., 2.]], requires_grad=True)
b1 = torch.tensor([0., -1.], requires_grad=True)
w2 = torch.tensor([1., 3.], requires_grad=True)
b2 = torch.tensor(0., requires_grad=True)

y_hat = torch.relu(x @ W1 + b1) @ w2 + b2
loss = (y_hat - 4) ** 2
loss.backward()

print("3.1")
print("y_hat =", y_hat.item(), " loss =", loss.item())
print("W1.grad =", W1.grad.tolist())
print("b1.grad =", b1.grad.tolist())
print("w2.grad =", w2.grad.tolist())
print("b2.grad =", b2.grad.item())

# XULOSA 3.1:
# Qo'lda hisob:
#   z1 = x @ W1 + b1 = (3, -1),  h = ReLU(z1) = (3, 0)
#   y_hat = 3*1 + 0*3 + 0 = 3,   loss = (3 - 4)^2 = 1
#   dL/dy_hat = 2*(3 - 4) = -2
#   w2.grad = h * (-2) = (-6, 0),   b2.grad = -2
#   dL/dh = w2 * (-2) = (-2, -6)
#   dL/dz1 = (-2, -6) * ReLU'(3, -1) = (-2, -6) * (1, 0) = (-2, 0)
#   b1.grad = (-2, 0),   W1.grad = outer(x, (-2, 0)) = [[-4, 0], [-2, 0]]
# PyTorch natijasi qo'lda hisob bilan to'liq mos keldi.
# Ikkinchi ustun gradienti 0, chunki z1 = -1 < 0 (o'lik ReLU).


#  3.2
belgilar = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=belgilar)
X = df[belgilar].values
y = df["species"].astype("category").cat.codes.values

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)


def fit_eval(Xtr, ytr, Xte, yte, lr=0.1, hid=8, log=False, zero_grad=True):
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    Xtr = torch.tensor(Xtr).float()
    Xte = torch.tensor(Xte).float()
    ytr = torch.tensor(ytr).long()
    yte = torch.tensor(yte).long()

    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(4, hid), nn.ReLU(), nn.Linear(hid, 3))
    opt = torch.optim.SGD(model.parameters(), lr=lr)

    for ep in range(201):
        loss = nn.functional.cross_entropy(model(Xtr), ytr)
        if zero_grad:
            opt.zero_grad()
        loss.backward()
        opt.step()
        if log and ep % 50 == 0:
            print("  epoch", ep, "loss", round(loss.item(), 4))

    acc = (model(Xte).argmax(1) == yte).float().mean()
    return round(acc.item(), 4)


# Jadval
print("\n3.2 Jadval (test aniqligi)")
print("hid \\ lr   0.01    0.1     1.0")
for hid in [2, 8, 32]:
    natija = []
    for lr in [0.01, 0.1, 1.0]:
        natija.append(fit_eval(Xtr, ytr, Xte, yte, lr=lr, hid=hid))
    print(hid, "      ", natija)

# XULOSA 3.2 (jadval):
# | hid \ lr | 0.01   | 0.1    | 1.0    |
# | 2        | 0.4369 | 0.8155 | 0.9806 |
# | 8        | 0.7961 | 0.9903 | 0.9903 |
# | 32       | 0.8641 | 0.9903 | 0.9903 |
#
# Tavsiya: hid = 8, lr = 0.1.
# Bu model hid = 32 bilan bir xil aniqlik beradi (0.9903, 103 tadan 102 tasi
# to'g'ri), lekin neyronlari 4 barobar kam, shuning uchun planshetda yengil
# ishlaydi. hid = 2 juda kichik: faqat katta lr = 1.0 bilan yaxshi natija
# beradi, kichik lr da esa yomon. lr = 0.1 esa lr = 1.0 dan xavfsizroq.


# zero_grad() xatosi
print("\nzero_grad() BILAN (hid=8, lr=0.1):")
print("  aniqlik:", fit_eval(Xtr, ytr, Xte, yte, log=True))

print("\nzero_grad() SIZ (hid=8, lr=0.1):")
print("  aniqlik:", fit_eval(Xtr, ytr, Xte, yte, log=True, zero_grad=False))

# XULOSA (zero_grad):
# zero_grad() bilan: loss 1.15 -> 0.27 -> 0.14 -> 0.09 -> 0.067, silliq tushadi.
# zero_grad() siz:  loss 1.15 -> 0.05 -> 0.0 -> 0.0 -> 0.0, juda tez "tushadi".
# PyTorch gradientlarni o'zi tozalamaydi, balki ularni qo'shib boradi.
# zero_grad() bo'lmasa, har qadamda gradient = barcha oldingi
# gradientlar yig'indisi bo'ladi. Qadamlar tobora kattalashadi va
# vaznlar haddan tashqari o'sib ketadi.

3.1
y_hat = 3.0  loss = 1.0
W1.grad = [[-4.0, 0.0], [-2.0, 0.0]]
b1.grad = [-2.0, 0.0]
w2.grad = [-6.0, -0.0]
b2.grad = -2.0

3.2 Jadval (test aniqligi)
hid \ lr   0.01    0.1     1.0
2        [0.4369, 0.8155, 0.9806]
8        [0.7961, 0.9903, 0.9903]
32        [0.8641, 0.9903, 0.9903]

zero_grad() BILAN (hid=8, lr=0.1):
  epoch 0 loss 1.15
  epoch 50 loss 0.2703
  epoch 100 loss 0.1403
  epoch 150 loss 0.0918
  epoch 200 loss 0.0667
  aniqlik: 0.9903

zero_grad() SIZ (hid=8, lr=0.1):
  epoch 0 loss 1.15
  epoch 50 loss 0.0505
  epoch 100 loss 0.0
  epoch 150 loss 0.0
  epoch 200 loss 0.0
  aniqlik: 0.9903


In [4]:
# ==================== 4-QISM: ISHDAGI VAZIYATLAR ====================

# 1. Tezroq model
# Aktivatsiyasiz qatlamlar faqat ko'paytirish amalini bajaradi.
# Masalan, (x*W1)*w2 = x*(W1*w2), ya'ni ikki qatlam aslida bitta qatlamga teng.
# 10 ta qatlam qo'shilsa ham, natija baribir bitta qatlam bo'lib qoladi
# va model faqat to'g'ri chiziq chiza oladi.
# Modelni aqlliroq qiladigan narsa aynan aktivatsiya.

# 2. Taksi narxi modeli
# Bu "o'lik ReLU" deb ataladi.
# Neyronga doim manfiy son kelsa, ReLU doim 0 chiqaradi va gradient ham 0 bo'ladi,
# shuning uchun neyron boshqa o'rganmaydi.
# Oldini olishning ikki usuli bor: learning rate'ni kichraytirish
# yoki Leaky ReLU ishlatish.

# 3. Xotira tejash
# Backward gradientni hisoblash uchun forward'da topilgan z1 va h qiymatlaridan foydalanadi.
# Masalan, dL/dw2 uchun h kerak, ReLU hosilasi uchun esa z1 kerak.
# Ular saqlanmasa, gradientni hisoblab bo'lmaydi
# yoki forward'ni qaytadan bajarishga to'g'ri keladi.

# 4. Kredit skoring
# Daromad millionlarda, yosh esa o'nlarda o'lchanadi,
# ya'ni sonlar masshtabi juda farq qiladi.
# Scaler'siz katta sonli belgi hammasini boshqarib qo'yadi
# va model hammaga bir xil javob beradi.
# Pingvinlarda ham shunday bo'lgan: scaler'siz model hammasini "Adelie" degan
# va aniqlik 99% dan 44% ga tushgan.

# 5. Loss = nan
# Formula: w <- w - lr * gradient.
# lr 100 barobar oshirilsa, har bir qadam ham 100 barobar kattalashadi
# va model minimumdan sakrab o'tib ketadi.
# Vaznlar tobora o'sib, oxiri juda katta sonlarga (inf) yetadi va hisob nan beradi.
# Yechim: lr'ni kichikroq qilish.

In [5]:
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


belgilar = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=belgilar)
X = df[belgilar].values
y = df["species"].astype("category").cat.codes.values   # 0, 1, 2

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Scaler
sc = StandardScaler().fit(Xtr)
Xtr = sc.transform(Xtr)
Xte = sc.transform(Xte)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)     # katta sonlardan himoya
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def one_hot(y, k=3):
    return np.eye(k)[y]


# MLP
class MLP:
    def __init__(self, n_in=4, n_h=8, n_out=3, seed=42):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 0.5, (n_in, n_h))
        self.b1 = np.zeros(n_h)
        self.W2 = r.normal(0, 0.5, (n_h, n_out))
        self.b2 = np.zeros(n_out)

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.h = np.maximum(0, self.z1)
        self.p = softmax(self.h @ self.W2 + self.b2)
        return self.p

    def step(self, X, y, lr):
        dz2 = (self.p - one_hot(y)) / len(X)
        dz1 = dz2 @ self.W2.T * (self.z1 > 0)
        self.W2 -= lr * self.h.T @ dz2
        self.b2 -= lr * dz2.sum(0)
        self.W1 -= lr * X.T @ dz1
        self.b1 -= lr * dz1.sum(0)


# O'qitish
net = MLP()
for ep in range(501):
    p = net.forward(Xtr)
    loss = -np.mean(np.log(p[np.arange(len(ytr)), ytr]))
    if ep % 100 == 0:
        print("epoch", ep, "loss", round(loss, 4))
    net.step(Xtr, ytr, lr=0.1)

bashorat = net.forward(Xte).argmax(axis=1)
aniqlik = (bashorat == yte).mean()
print("Test aniqligi:", round(aniqlik, 4))
print("Qabul qilindi!" if aniqlik > 0.95 else "Hali yetarli emas")

# XULOSA:
# Loss 1.43 dan 0.04 gacha tushdi, test aniqligi 0.9903
# (103 tadan 102 tasi to'g'ri), bu 0.95 dan yuqori.
# PyTorch'siz, faqat NumPy bilan yozilgan model PyTorch modeli bilan
# bir xil aniqlik berdi. Softmax + cross-entropy uchun chiqishdagi
# gradient oddiy: (p - one_hot(y)) / N. Scaler bo'lmasa model ishlamaydi.

epoch 0 loss 1.4278
epoch 100 loss 0.1903
epoch 200 loss 0.1025
epoch 300 loss 0.0661
epoch 400 loss 0.0495
epoch 500 loss 0.0398
Test aniqligi: 0.9903
Qabul qilindi!
